Generator kya hota hai?
Generator ek special function hota hai Python mein jo ek iterator return karta hai. Yeh saari values ek saath calculate nahi karta — balki ek ek karke value yield karta hai, aur beech mein ruk jaata hai. Jab agli value maango, tab wahan se shuru ho jaata hai jahan rukha tha.

yield keyword se generator banana padta hai. Jis bhi function mein ek bhi yield ho, woh function automatically generator function ban jaata hai.

In [ ]:
# Basic Syntax (likhne ka tarika)
def my_generator():
    yield 1     # pauses here, returns 1
    yield 2     # pauses here, returns 2
    yield 3     # pauses here, returns 3

gen = my_generator()   # creates generator object
print(next(gen))         # 1
print(next(gen))         # 2
print(next(gen))         # 3
# yield matlab "ruko aur bhejo" — function apni state yaad rakhta hai har call ke beech mein.

1
2


Execution Flow
How does a generator execute?
1

Call the function → Python doesn't run the body yet. It just creates a generator object.
2

Call next() → The body starts running until the first yield is hit.
3

yield hits → The yielded value is returned, and the function is frozen at that line.
4

next() again → Execution resumes from exactly after the last yield.
5

Function ends → Python raises StopIteration automatically.

Memory
Asli baat: lazy evaluation
Generator values tab banata hai jab maango. 10 lakh numbers ka generator almost zero memory leta hai — kyunki saare numbers stored nahi hote, sirf current state hoti hai.

In [4]:
def square(n):
    for i in range(3):
        yield i**2

square(3)

<generator object square at 0x000002CD07D3A810>

In [5]:
for i in square(3):
    print(i)

0
1
4


In [14]:
a = square(3)
next(a)
next(a)
next(a)

4

In [20]:
def my_generator():
    yield 1
    yield 2
    yield 3

gen = my_generator()
gen
next(gen)

1

In [21]:
for val in gen:
    print(val)

2
3


`Generators are particularly used for reading large files because they allow you to process  one line at a time without loading the entire file into memory

10 GB ki log file — bina RAM bhare padhna
Production server pe 10 GB log file hai. List mein load karte toh RAM full. Generator se ek line, process, agli line — memory almost zero.

In [ ]:
# practical : reading large files line by line
file_path = "server.log"
def read_logs(file_path):
    with open(file_path, 'r') as f:
        for line in f:
            yield line.strip()  #ek line do, ruko


# use it
for line in read_logs(file_path):
    print(line)

2024-01-15 09:00:01 INFO  GET /api/users 200 OK 12ms
2024-01-15 09:00:03 ERROR DB connection timeout after 30s
2024-01-15 09:00:05 INFO  POST /api/login 200 OK 34ms
2024-01-15 09:00:07 WARN  High memory usage: 87%
2024-01-15 09:00:09 ERROR NullPointerException in OrderService.py line 142
2024-01-15 09:00:11 INFO  GET /api/products 200 OK 8ms
2024-01-15 09:00:13 INFO  User 4821 logged in from 192.168.1.1
2024-01-15 09:00:15 ERROR Payment gateway timeout — retry 1/3
2024-01-15 09:00:17 INFO  Cache hit ratio: 94%
2024-01-15 09:00:19 WARN  Slow query detected — 2.3s on users table
2024-01-15 09:00:21 INFO  Scheduled backup job completed
2024-01-15 09:00:23 ERROR SSL certificate expiring in 3 days
2024-01-15 09:00:25 INFO  GET /dashboard 200 OK 22ms
2024-01-15 09:00:27 WARN  Rate limit approaching for IP 10.0.0.5
2024-01-15 09:00:29 ERROR Unhandled exception: FileNotFoundException
2024-01-15 09:00:31 INFO  New user registered id=9912
2024-01-15 09:00:33 INFO  GET /api/orders 200 OK 18ms
202

I use generator pipelines to process large log files lazily, chaining read → filter → parse steps without loading the entire dataset into memory.”

In [24]:
# practical : reading large files line by line

file_path = "server.log"

# --- Step 1: Read logs ---
def read_logs(file_path):
    with open(file_path, 'r') as f:
        for line in f:
            yield line.strip()


# --- Step 2: Filter ERROR logs ---
def filter_errors(logs):   # ✅ fixed name
    for log in logs:
        if "ERROR" in log:
            yield log


# --- Step 3: Parse logs ---
def parse_logs(logs):
    for log in logs:
        parts = log.split(" ", 3)
        yield {
            "date": parts[0],
            "time": parts[1],
            "level": parts[2],
            "message": parts[3]
        }


# --- Step 4: Pipeline ---
pipeline = parse_logs(
              filter_errors(
                  read_logs(file_path)
              )
          )


# --- Step 5: Output parsed ERROR logs ---
for entry in pipeline:
    print(f"[{entry['time']}] {entry['level']} → {entry['message']}")


# --- Step 6: (Optional) Print all logs ---
print("\nAll logs:")
for line in read_logs(file_path):
    print(line)

[09:00:03] ERROR → DB connection timeout after 30s
[09:00:09] ERROR → NullPointerException in OrderService.py line 142
[09:00:15] ERROR → Payment gateway timeout — retry 1/3
[09:00:23] ERROR → SSL certificate expiring in 3 days
[09:00:29] ERROR → Unhandled exception: FileNotFoundException
[09:00:35] ERROR → Memory limit exceeded — process killed

All logs:
2024-01-15 09:00:01 INFO  GET /api/users 200 OK 12ms
2024-01-15 09:00:03 ERROR DB connection timeout after 30s
2024-01-15 09:00:05 INFO  POST /api/login 200 OK 34ms
2024-01-15 09:00:07 WARN  High memory usage: 87%
2024-01-15 09:00:09 ERROR NullPointerException in OrderService.py line 142
2024-01-15 09:00:11 INFO  GET /api/products 200 OK 8ms
2024-01-15 09:00:13 INFO  User 4821 logged in from 192.168.1.1
2024-01-15 09:00:15 ERROR Payment gateway timeout — retry 1/3
2024-01-15 09:00:17 INFO  Cache hit ratio: 94%
2024-01-15 09:00:19 WARN  Slow query detected — 2.3s on users table
2024-01-15 09:00:21 INFO  Scheduled backup job completed
